# Memory AI Lab — Évaluation ARI V4

**Workflow :** éditer le code dans VS Code → `git push` → ouvrir ce notebook dans [colab.research.google.com](https://colab.research.google.com)

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Protocole anti-surapprentissage

```
group_gold_tune.json  → 448 épisodes, 8954 msgs  (août 2023 → jan 2025)
                         ↑ Optuna bayésien — 20 trials
                           Seuls attach_threshold, ema_alpha, time_threshold sont tunés.
                           hard_break et dormancy = valeurs sémantiques fixes.

group_gold_test.json  → 193 épisodes, 2786 msgs  (jan 2025 → mars 2026)
                         ↑ SCORE FINAL — une seule fois, ne pas tuner dessus
```

## Modes disponibles

- `USE_HYBRID = False` : EpisodeSegmenterFast seul (rapide, baseline)
- `USE_HYBRID = True`  : HybridEpisodeSegmenter (Stage 1+2+3) — nécessite boundary_detector.pt

Pour entraîner le boundary detector : ouvrir `02_train_boundary_detector.ipynb`

**Données requises sur Google Drive (`memory_ai_data/`) :**
```
group_anon.txt
group_gold_tune.json
group_gold_test.json
boundary_detector.pt   (optionnel — requis si USE_HYBRID=True)
```

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
# Fix sympy/torch incompatibilité — DOIT être installé avant tout import torch
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!pip install optuna -q
!python -m spacy download fr_core_news_sm -q
print('✓ OK')
print()
print('⚠️  Si première exécution : Exécution > Redémarrer la session,')
print('   puis relancer à partir de la cellule 3 (les imports sont en cache).')

In [ ]:
# ── CELLULE 3 : Google Drive + Copie locale (évite les coupures Drive) ───
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/memory_ai_data'
LOCAL_DIR = '/content/data'
os.makedirs(LOCAL_DIR, exist_ok=True)

# Fichiers statiques — copiés une seule fois (skip si déjà en local)
STATIC_FILES = [
    'group_anon.txt',
    'group_gold_tune.json',
    'group_gold_test.json',
    'group_embeddings_me5.npy',
]
# Fichiers volatils — toujours recopiés (peuvent être régénérés entre sessions)
VOLATILE_FILES = [
    'boundary_detector.pt',
]

for fname in STATIC_FILES:
    src, dst = f'{DRIVE_DIR}/{fname}', f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        print(f'  Copie {fname} ...', end=' ', flush=True)
        shutil.copy2(src, dst)
        print('✓')
    elif os.path.exists(dst):
        print(f'  {fname} déjà en local ✓')
    else:
        print(f'  {fname} absent sur Drive (ignoré)')

for fname in VOLATILE_FILES:
    src, dst = f'{DRIVE_DIR}/{fname}', f'{LOCAL_DIR}/{fname}'
    if os.path.exists(src):
        print(f'  Copie {fname} (volatile) ...', end=' ', flush=True)
        shutil.copy2(src, dst)
        print('✓')
    else:
        print(f'  {fname} absent sur Drive (ignoré)')

DATA_DIR = LOCAL_DIR

# ── Mode hybride ─────────────────────────────────────────────────────────
USE_HYBRID = True   # True = HybridEpisodeSegmenter Stage 1+2+3

print(f'\n✓ DATA_DIR = {DATA_DIR}')
print(f'  Mode : {"Hybride (Stage 1+2+3)" if USE_HYBRID else "Fast (Stage 2+3 only)"}')

In [ ]:
# ── CELLULE 4 : Parse + Embeddings mE5-base (GPU + cache) ─────────────────
import numpy as np
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from parsers.whatsapp_parser import parse_whatsapp_chat

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

# mE5-base — multilingual, 768d, handles FR/EN code-switching
# Remplace all-MiniLM-L6-v2 (384d, EN only)
MODEL_NAME  = 'intfloat/multilingual-e5-base'
PREFIX      = 'passage: '   # requis par mE5 pour les documents
EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings_me5.npy'

all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [PREFIX + a.content for a in all_artifacts]
print(f'[1/2] {len(texts)} messages parsés')

if EMBED_CACHE.exists():
    all_embeddings = np.load(EMBED_CACHE)
    assert len(all_embeddings) == len(texts), 'Cache périmé — supprimer group_embeddings_me5.npy'
    print('[2/2] Embeddings chargés depuis cache')
else:
    print(f'[2/2] Calcul sur {device} (mE5-base 768d) ...')
    model = SentenceTransformer(MODEL_NAME, device=device)
    all_embeddings = model.encode(
        texts, batch_size=256, show_progress_bar=True,
        device=device, convert_to_numpy=True
    ).astype(np.float32)
    np.save(EMBED_CACHE, all_embeddings)
    print(f'      Sauvegardé → {EMBED_CACHE}')

print(f'      Shape : {all_embeddings.shape}  (attendu : (n, 768))')

In [ ]:
# ── CELLULE 5 : Charger tune / test avec split B+C ────────────────────────
# TUNE_SPLIT doit être identique à 02_train_boundary_detector.ipynb
TUNE_SPLIT = 0.65   # 65% tune_early → boundary detector | 35% tune_late → Optuna

import json

def load_split(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    n = len(data['artifacts'])
    y_true = [None] * n
    for ep in data['episodes']:
        for idx in range(ep['start_idx'], ep['end_idx'] + 1):
            if idx < n:
                y_true[idx] = ep['episode_id']
    return data['artifacts'], y_true, data['episodes'], data['meta']

tune_arts_all, y_true_tune_all, tune_eps_all, tune_meta = load_split(f'{DATA_DIR}/group_gold_tune.json')
test_arts,     y_true_test,     test_eps,     test_meta  = load_split(f'{DATA_DIR}/group_gold_test.json')

n_tune  = len(tune_arts_all)
n_split = int(n_tune * TUNE_SPLIT)

arts_tune_all = all_artifacts[:n_tune]
arts_test     = all_artifacts[n_tune:n_tune + len(test_arts)]
emb_test      = all_embeddings[n_tune:n_tune + len(test_arts)]

# tune_late — données vierges pour Optuna (jamais vues par le boundary detector)
arts_tune_late = arts_tune_all[n_split:]
emb_tune_late  = all_embeddings[n_split:n_tune]
y_true_late    = y_true_tune_all[n_split:]

# tune_all — pour l'évaluation de référence en cellule 9
emb_tune_all   = all_embeddings[:n_tune]

print(f'Tune total : {n_tune} msgs · {len(tune_eps_all)} épisodes · {tune_meta["period"]}')
print(f'Tune early : {n_split} msgs  → boundary detector (02_train_boundary_detector.ipynb)')
print(f'Tune late  : {n_tune - n_split} msgs  → Optuna (cette session)')
print(f'Test       : {len(test_arts)} msgs · {len(test_eps)} épisodes · {test_meta["period"]}')

In [ ]:
# ── CELLULE 6 : Helper d'évaluation ───────────────────────────────────────
import subprocess, importlib, sys
for pkg in ['optuna', 'cmaes']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Force reload des modules locaux après git pull
# IMPORTANT : models doit être rechargé EN PREMIER (les autres en dépendent)
for mod in ['models',
            'boundary_detector',
            'episode_algorithm', 'episode_algorithm_fast',
            'episode_segmenter_hybrid',
            'episode_splitter', 'episode_splitter_fast',
            'episode_merger']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from episode_algorithm_fast import EpisodeSegmenterFast
from episode_splitter_fast import EpisodeSplitterFast, SplitConfig
from episode_merger import EpisodeMerger
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

# ── Boundary Detector (Stage 1) ───────────────────────────────────────────
DETECTOR = None
if USE_HYBRID:
    from boundary_detector import BoundaryDetector
    DETECTOR_PATH = f'{DATA_DIR}/boundary_detector.pt'
    import os
    if os.path.exists(DETECTOR_PATH):
        DETECTOR = BoundaryDetector(device=DEVICE)
        DETECTOR.load(DETECTOR_PATH)
        print(f'✓ Boundary detector chargé (seuil={DETECTOR.threshold:.2f})')
    else:
        print('⚠️  boundary_detector.pt introuvable — exécuter 02_train_boundary_detector.ipynb d\'abord')
        USE_HYBRID = False

SPLITTER = EpisodeSplitterFast(
    SplitConfig(
        min_cohesion=0.65,
        min_size_to_split=8,
        max_span_hours=168.0,
        max_splits=6,
        min_sub_size=3,
    ),
    device=DEVICE,
    quality_threshold=0.10,
)

FIXED_PARAMS = dict(
    alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
    dormancy_minutes=1440,
    hard_break_minutes=0,
    active_penalty_hours=24.0,
    allow_reactivation=True,
)

MERGER_KEYS = {'min_size', 'merge_sim', 'merge_gap_minutes'}
HYBRID_KEYS = {'boundary_k', 'bd_threshold'}

def run_eval(artifacts, embeddings, y_true, tunable_params, verbose=False, use_splitter=False):
    seg_params    = {k: v for k, v in tunable_params.items() if k not in MERGER_KEYS}
    merger_params = {k: v for k, v in tunable_params.items() if k in MERGER_KEYS}
    params        = {**FIXED_PARAMS, **seg_params}

    if USE_HYBRID and DETECTOR is not None:
        from episode_segmenter_hybrid import HybridEpisodeSegmenter
        seg = HybridEpisodeSegmenter(detector=DETECTOR, device=DEVICE, **params)
    else:
        seg = EpisodeSegmenterFast(device=DEVICE, **{k: v for k, v in params.items() if k not in HYBRID_KEYS})

    eps = seg.consolidate(seg.segment(artifacts, embeddings))
    if use_splitter:
        eps = SPLITTER.split(eps, artifacts, embeddings, verbose=False)

    if merger_params:
        merger = EpisodeMerger(device=DEVICE, **merger_params)
        eps = merger.merge(eps)

    n = len(artifacts)
    y_pred = [None] * n
    for ep in eps:
        for idx in ep.artifact_indices:
            if idx < n:
                y_pred[idx] = ep.id

    pairs = [(t, p) for t, p in zip(y_true, y_pred) if t is not None and p is not None]
    if not pairs:
        return 0.0, 0.0, len(eps)
    yt, yp = zip(*pairs)
    ari = adjusted_rand_score(yt, yp)
    nmi = normalized_mutual_info_score(yt, yp)
    if verbose:
        n_gold = len(set(t for t in y_true if t is not None))
        mode = 'hybrid+merge' if (USE_HYBRID and DETECTOR and merger_params) else \
               'hybrid' if (USE_HYBRID and DETECTOR) else 'fast'
        print(f'  [{mode}] ARI={ari:+.4f}  NMI={nmi:.4f}  gold={n_gold}  pred={len(eps)}')
    return ari, nmi, len(eps)

print('✓ Helpers prêts')

In [ ]:
# ── CELLULE 7 : Optimisation bayésienne sur TUNE_LATE ─────────────────────
# tune_late : pas de leak boundary detector (entraîné sur tune_early)
#   Stage 1 : bd_threshold
#   Stage 2 : attach_threshold, ema_alpha, time_threshold, boundary_k
#   Merger  : min_size, merge_sim [0.50,0.95], merge_gap — dans Optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_FAST   = min(1000, len(arts_tune_late))
N_TRIALS = 160
N_JOBS   = 2

arts_fast = arts_tune_late[:N_FAST]
emb_fast  = emb_tune_late[:N_FAST]
y_fast    = y_true_late[:N_FAST]
print(f'Tuning {N_TRIALS} trials × {N_FAST} msgs (tune_late) | n_jobs={N_JOBS} | device={DEVICE}')

sampler = optuna.samplers.CmaEsSampler(seed=42, n_startup_trials=10)

def objective(trial):
    params = dict(
        attach_threshold       = trial.suggest_float('attach',           0.30, 0.70),
        ema_alpha              = trial.suggest_float('ema',              0.50, 0.95),
        time_threshold_minutes = trial.suggest_int  ('time_thr',        60, 480, step=30),
        # Merger — range étendu : Optuna choisissait 0.877 sur [0.40,0.90]
        min_size               = trial.suggest_int  ('min_size',         1,   6),
        merge_sim              = trial.suggest_float('merge_sim',        0.50, 0.97),
        merge_gap_minutes      = trial.suggest_float('merge_gap',        15., 180.),
    )
    if USE_HYBRID:
        params['boundary_k']   = trial.suggest_float('boundary_k',   0.0,  0.5)
        params['bd_threshold'] = trial.suggest_float('bd_threshold', 0.10, 0.80)
    ari, _, _ = run_eval(arts_fast, emb_fast, y_fast, params)
    return ari

study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=N_JOBS,
    callbacks=[lambda s, t: print(f'  Trial {t.number:3d} | ARI={t.value:+.4f} | {t.params}')],
)

raw = study.best_params
best_params = dict(
    attach_threshold       = raw['attach'],
    ema_alpha              = raw['ema'],
    time_threshold_minutes = raw['time_thr'],
    min_size               = raw['min_size'],
    merge_sim              = raw['merge_sim'],
    merge_gap_minutes      = raw['merge_gap'],
)
if USE_HYBRID:
    best_params['boundary_k']   = raw.get('boundary_k', 0.1)
    best_params['bd_threshold'] = raw.get('bd_threshold', DETECTOR.threshold if DETECTOR else 0.3)

print(f'\n✓ Meilleurs paramètres (CMA-ES, sur tune_late) :')
for k, v in best_params.items():
    print(f'  {k} = {v:.4f}' if isinstance(v, float) else f'  {k} = {v}')
print(f'  ARI tune_late = {study.best_value:+.4f}')

In [ ]:
# ── CELLULE 8 : Importance des paramètres ─────────────────────────────────
importance = optuna.importance.get_param_importances(study)
print('Importance des paramètres :')
for k, v in importance.items():
    bar = '█' * int(v * 40)
    print(f'  {k:25s} {bar} {v:.3f}')

import pandas as pd
cols = ['number','value','params_attach','params_ema','params_time_thr','params_merge_sim']
df_trials = study.trials_dataframe()[cols]
df_trials.columns = ['trial', 'ari', 'attach', 'ema', 'time_thr', 'merge_sim']
print(f'\nTop 5 trials :')
print(df_trials.sort_values('ari', ascending=False).head(5).to_string(index=False))

In [ ]:
# ── CELLULE 9 : Score FINAL sur TEST ──────────────────────────────────────
# Exécuter UNE SEULE FOIS. Ne pas re-run après avoir vu le score.

raw = study.best_params
best_params = dict(
    attach_threshold       = raw['attach'],
    ema_alpha              = raw['ema'],
    time_threshold_minutes = raw['time_thr'],
    min_size               = raw['min_size'],
    merge_sim              = raw['merge_sim'],
    merge_gap_minutes      = raw['merge_gap'],
)
if USE_HYBRID:
    best_params['boundary_k']   = raw.get('boundary_k', 0.1)
    best_params['bd_threshold'] = raw.get('bd_threshold', DETECTOR.threshold if DETECTOR else 0.3)

print('Évaluation TUNE ALL (référence) :')
ari_tune, nmi_tune, n_pred_tune = run_eval(
    arts_tune_all, emb_tune_all, y_true_tune_all, best_params, verbose=True)

print('\nÉvaluation TEST (officiel) :')
ari_test, nmi_test, n_pred_test = run_eval(
    arts_test, emb_test, y_true_test, best_params, verbose=True)

gap = ari_tune - ari_test
gap_status = '✓ OK — bonne généralisation' if abs(gap) < 0.05 else '⚠️  Écart élevé — vérifier'
mode_label = 'V4 C HYBRID+MERGE' if (USE_HYBRID and DETECTOR) else 'V4 C FAST+MERGE'

print(f"""
╔══════════════════════════════════════════════════╗
║  RÉSULTAT OFFICIEL {mode_label:<28} ║
╠══════════════════════════════════════════════════╣
║  ARI  (test)   : {ari_test:+.4f}                 ║
║  NMI  (test)   : {nmi_test:.4f}                  ║
║  Gold (test)   : {len(test_eps)} épisodes         ║
║  Pred (test)   : {n_pred_test} épisodes           ║
╠══════════════════════════════════════════════════╣
║  ARI  (tune)   : {ari_tune:+.4f}                 ║
║  Gap  tune-test: {gap:+.4f}  {gap_status}   ║
╚══════════════════════════════════════════════════╝

Protocole B+C :
  Boundary detector entraîné sur tune_early ({int(n_tune*TUNE_SPLIT)} msgs)
  Optuna tourné sur tune_late ({n_tune - int(n_tune*TUNE_SPLIT)} msgs)
  Score final sur test ({len(test_arts)} msgs) — aucune donnée partagée
""")
print('Paramètres finaux :')
for k, v in {**FIXED_PARAMS, **best_params}.items():
    print(f'  {k} = {v}')